# 🚀 Step 2: Train RLOO (REINFORCE Leave-One-Out)\n⚠️ **WARNING: HIGH VRAM USAGE!**\nThis will load the Policy Model + Reference Model + Reward Model simultaneously. It may OOM on an 8GB GPU.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer, BitsAndBytesConfig
from peft import get_peft_model, LoraConfig, TaskType
from datasets import load_dataset
from trl import RLOOTrainer, RLOOConfig

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
base_model_name = "Qwen/Qwen2.5-3B-Instruct"

# 1. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 2. Load Policy Model (to be trained)
policy_model = AutoModelForCausalLM.from_pretrained(
    "../01_sft/qwen_medical_arabic_lora", # Start from SFT
    quantization_config=bnb_config,
    device_map="auto"
)
lora_config = LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"], bias="none")
policy_model = get_peft_model(policy_model, lora_config)

# 3. Load Reference Model (frozen SFT model)
ref_model = AutoModelForCausalLM.from_pretrained(
    "../01_sft/qwen_medical_arabic_lora",
    quantization_config=bnb_config,
    device_map="auto"
)
ref_model.eval()

# 4. Load Reward Model (trained in part 1)
reward_model = AutoModelForSequenceClassification.from_pretrained(
    "qwen_medical_arabic_reward_model",
    num_labels=1,
    quantization_config=bnb_config,
    device_map="auto"
)
reward_model.config.pad_token_id = tokenizer.pad_token_id
reward_model.eval()

print("All 3 models loaded! (Check your VRAM)")


In [ ]:
# Prepare Prompt-only dataset for RLOO
dataset = load_dataset("json", data_files="../../data/alignment/05_dpo_dataset.json", split="train")

SYSTEM_MSG = "أنت معالج نفسي عربي خبير. ردودك آمنة ومتعاطفة وتراعي التعاليم الإسلامية السنية. لا تشخص ولا تصف أدوية."

def format_rloo(example):
    prompt = f"<|im_start|>system\n{SYSTEM_MSG}<|im_end|>\n<|im_start|>user\n{example['prompt']}<|im_end|>\n<|im_start|>assistant\n"
    return {"prompt": prompt}

dataset = dataset.map(format_rloo, remove_columns=dataset.column_names)


In [ ]:
training_args = RLOOConfig(
    output_dir="outputs_rloo",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-5, # RLOO requires smaller LR
    max_length=1024, # Max total length
    optim="adamw_8bit",
    num_train_epochs=1,
    rloo_k=2, # Generate 2 responses per prompt (Leave-One-Out uses k-1 as baseline)
    logging_steps=5,
    report_to="none",
)

trainer = RLOOTrainer(
    model=policy_model,
    ref_model=ref_model,
    reward_model=reward_model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
)

print("Starting RLOO training... (May cause OOM on 8GB VRAM)")
trainer.train()

policy_model.save_pretrained("qwen_medical_arabic_rloo")
tokenizer.save_pretrained("qwen_medical_arabic_rloo")
print("RLOO model saved!")
